# Assessment 3 — SQLite Database Persistence for Credit Risk Analysis

**Student:** Sanjoy Bose  
**Course:** 161.777 Practical Data Mining  
**Deadline:** 31 May 2026

---

This notebook covers all SQLite database requirements for the credit risk project: schema design, normalised table creation, data population, join-based queries, views, and database verification. The data preparation and EDA work is in the separate Assessment 4 notebook.

**Run the Assessment 4 notebook first** to generate the required CSV files before running this notebook.


## 1. Import Libraries


In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


## 2. Read Prepared Files from Assessment 4

These CSV files are produced by the Assessment 4 notebook. If they are not available, run the Assessment 4 notebook from top to bottom first.


In [ ]:
credit_file = "prepared_credit_risk.csv"
macro_file = "prepared_worldbank_macro.csv"
rbnz_file = "prepared_rbnz_ocr.csv"

for f in [credit_file, macro_file]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"{f} not found. Please run Assessment 4 notebook first.")

credit_clean = pd.read_csv(credit_file)
macro_wide = pd.read_csv(macro_file)
rbnz_annual = pd.read_csv(rbnz_file) if os.path.exists(rbnz_file) else pd.DataFrame()

print("Credit data shape:", credit_clean.shape)
print("Macro data shape:", macro_wide.shape)
print("RBNZ data shape:", rbnz_annual.shape)
display(credit_clean.head(3))
display(macro_wide.head(3))


## 3. Database Schema Design

### 3.1 Schema Description

The database is designed to store credit risk data in a normalised relational structure. Normalisation avoids duplication and makes the data easier to query and maintain. The design follows third normal form (3NF) — each non-key attribute depends only on the primary key of its table.

The schema contains **six tables**:

| Table | Purpose | Primary Key | Foreign Keys |
|---|---|---|---|
| `borrowers` | Borrower demographic and income information | `applicant_id` | `assessment_year` → `macro_indicators.year` |
| `loans` | Loan amount, purpose, grade and interest rate | `loan_id` | `applicant_id` → `borrowers`, `source_id` → `data_sources` |
| `credit_history` | Previous default and credit history details | `history_id` | `applicant_id` → `borrowers` |
| `risk_outcomes` | Current default flag and descriptive risk band | `outcome_id` | `applicant_id` → `borrowers` |
| `macro_indicators` | World Bank and RBNZ macroeconomic indicators | `macro_id` | — |
| `data_sources` | Metadata about the three data sources used | `source_id` | — |

**Relationships:**
- `borrowers` is the central entity. Each borrower has one record in `loans`, one in `credit_history`, and one in `risk_outcomes`.
- `borrowers.assessment_year` links to `macro_indicators.year` to associate each borrower with macroeconomic conditions.
- `loans.source_id` links to `data_sources` to record which source the loan data came from.
- `macro_indicators` stores both World Bank and RBNZ indicators merged into one table to simplify joining.
- `data_sources` documents all three acquisition sources (Kaggle static CSV, World Bank API, RBNZ web scraping).

**Design decisions:**
- Indexes are created on all foreign key columns and frequently queried columns (loan grade, risk band) to improve query performance.
- CHECK constraints enforce business rules: age must be 18–100, income and loan amounts must be non-negative, default flag must be 0 or 1.
- UNIQUE constraint on `macro_indicators.year` prevents duplicate yearly rows.


### 3.2 Schema Diagram

The diagram below shows the six tables, their attributes, primary keys (PK), and foreign key relationships (FK). It is generated programmatically using matplotlib and saved as `credit_risk_schema.png`.

**For a more polished ERD**, this image can be replaced with a diagram exported from DBSchema (https://dbschema.com/) using the same table and relationship structure described above.


In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.axis("off")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

box_style = dict(boxstyle="round,pad=0.4", edgecolor="#333333", facecolor="#f0f4ff", linewidth=1.5)
pk_style = dict(boxstyle="round,pad=0.4", edgecolor="#0055aa", facecolor="#ddeeff", linewidth=2)

tables = {
    "borrowers": (
        0.08, 0.55,
        "borrowers\n─────────────────\nPK  applicant_id\n    person_age\n    person_income\n    person_home_ownership\n    person_emp_length\nFK  assessment_year → macro_indicators.year"
    ),
    "loans": (
        0.40, 0.82,
        "loans\n─────────────────\nPK  loan_id\nFK  applicant_id → borrowers\nFK  source_id → data_sources\n    loan_intent\n    loan_grade\n    loan_amnt\n    loan_int_rate\n    loan_percent_income"
    ),
    "credit_history": (
        0.40, 0.50,
        "credit_history\n─────────────────\nPK  history_id\nFK  applicant_id → borrowers\n    cb_person_default_on_file\n    cb_person_cred_hist_length\n    credit_score"
    ),
    "risk_outcomes": (
        0.40, 0.20,
        "risk_outcomes\n─────────────────\nPK  outcome_id\nFK  applicant_id → borrowers\n    default_flag  (0/1)\n    risk_band"
    ),
    "macro_indicators": (
        0.75, 0.65,
        "macro_indicators\n─────────────────\nPK  macro_id\n    country_code\n    year  (UNIQUE)\n    inflation_pct\n    unemployment_pct\n    lending_rate_pct\n    domestic_credit_pct_gdp\n    avg_ocr_pct  (RBNZ)"
    ),
    "data_sources": (
        0.75, 0.22,
        "data_sources\n─────────────────\nPK  source_id\n    source_name\n    source_type\n    description"
    ),
}

for name, (x, y, label) in tables.items():
    ax.text(x, y, label, ha="center", va="center", fontsize=8.5,
            fontfamily="monospace", bbox=box_style, transform=ax.transAxes)

arrowprops = dict(arrowstyle="->", color="#555555", linewidth=1.5)
# borrowers → loans
ax.annotate("", xy=(0.26, 0.82), xytext=(0.16, 0.62), arrowprops=arrowprops, xycoords="axes fraction", textcoords="axes fraction")
# borrowers → credit_history
ax.annotate("", xy=(0.26, 0.50), xytext=(0.16, 0.55), arrowprops=arrowprops, xycoords="axes fraction", textcoords="axes fraction")
# borrowers → risk_outcomes
ax.annotate("", xy=(0.26, 0.20), xytext=(0.16, 0.48), arrowprops=arrowprops, xycoords="axes fraction", textcoords="axes fraction")
# borrowers → macro_indicators
ax.annotate("", xy=(0.62, 0.65), xytext=(0.18, 0.60), arrowprops=arrowprops, xycoords="axes fraction", textcoords="axes fraction")
# loans → data_sources
ax.annotate("", xy=(0.62, 0.30), xytext=(0.55, 0.75), arrowprops=dict(arrowstyle="->", color="#aa5500", linewidth=1.2, linestyle="dashed"), xycoords="axes fraction", textcoords="axes fraction")

ax.text(0.50, 0.97, "Credit Risk SQLite Database Schema",
        ha="center", va="center", fontsize=15, weight="bold", transform=ax.transAxes)
ax.text(0.50, 0.93, "Six tables | PK = Primary Key | FK = Foreign Key | Solid arrows = FK relationships",
        ha="center", va="center", fontsize=9, color="#555555", transform=ax.transAxes)

schema_image = "credit_risk_schema.png"
plt.savefig(schema_image, bbox_inches="tight", dpi=150)
plt.show()
display(Image(filename=schema_image))


## 4. Prepare Normalised DataFrames

The flat credit dataset is split into the six normalised tables before being loaded into SQLite.


In [ ]:
# Table 1: borrowers
borrowers = credit_clean[[
    "applicant_id", "person_age", "person_income",
    "person_home_ownership", "person_emp_length", "assessment_year"
]].copy()

# Table 2: loans
loans = credit_clean[[
    "applicant_id", "loan_intent", "loan_grade", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "loan_to_income_check"
]].copy()
loans.insert(0, "loan_id", np.arange(1, len(loans) + 1))
loans["source_id"] = 1  # Kaggle static source

# Table 3: credit_history
credit_history = credit_clean[[
    "applicant_id", "cb_person_default_on_file", "cb_person_cred_hist_length", "credit_score"
]].copy()
credit_history.insert(0, "history_id", np.arange(1, len(credit_history) + 1))

# Table 4: risk_outcomes
risk_outcomes = credit_clean[[
    "applicant_id", "default_flag", "risk_band"
]].copy()
risk_outcomes.insert(0, "outcome_id", np.arange(1, len(risk_outcomes) + 1))

# Table 5: macro_indicators (World Bank + RBNZ merged)
macro_indicators = macro_wide.copy()
if len(rbnz_annual) > 0 and "avg_ocr_pct" in rbnz_annual.columns:
    macro_indicators = pd.merge(
        macro_indicators,
        rbnz_annual[["year", "avg_ocr_pct"]],
        on="year",
        how="left"
    )
else:
    macro_indicators["avg_ocr_pct"] = np.nan
macro_indicators.insert(0, "macro_id", np.arange(1, len(macro_indicators) + 1))

# Table 6: data_sources — documents all three acquisition sources
data_sources = pd.DataFrame({
    "source_id": [1, 2, 3],
    "source_name": [
        "Credit Risk Benchmark Dataset",
        "World Bank Indicators API",
        "RBNZ OCR History (Web Scraping)"
    ],
    "source_type": ["Static CSV", "Dynamic API", "Dynamic Web Scraping"],
    "description": [
        "Borrower and loan-level credit risk data downloaded from Kaggle as a CSV file.",
        "Macroeconomic indicators (inflation, unemployment, lending rate, domestic credit) retrieved from the public World Bank API for New Zealand 2018-2024.",
        "Official Cash Rate history scraped from the RBNZ website using BeautifulSoup."
    ]
})

print("Table shapes:")
for name, df in [("borrowers", borrowers), ("loans", loans), ("credit_history", credit_history),
                  ("risk_outcomes", risk_outcomes), ("macro_indicators", macro_indicators), ("data_sources", data_sources)]:
    print(f"  {name}: {df.shape}")


## 5. Create SQLite Database and Schema

The database is created fresh each time the notebook runs. Tables are dropped in reverse dependency order first so that foreign key constraints are respected. All primary keys, foreign keys, CHECK constraints and indexes are defined explicitly in the `CREATE TABLE` statements rather than relying on `to_sql` defaults.


In [ ]:
db_name = "credit_risk_assignment3.sqlite"
conn = sqlite3.connect(db_name)
cur = conn.cursor()

cur.execute("PRAGMA foreign_keys = ON;")

# Drop tables in reverse dependency order
drop_order = ["risk_outcomes", "credit_history", "loans", "borrowers", "macro_indicators", "data_sources"]
for table in drop_order:
    cur.execute(f"DROP TABLE IF EXISTS {table};")

cur.executescript("""

CREATE TABLE data_sources (
    source_id   INTEGER PRIMARY KEY,
    source_name TEXT    NOT NULL,
    source_type TEXT    NOT NULL
                        CHECK (source_type IN ('Static CSV', 'Dynamic API', 'Dynamic Web Scraping')),
    description TEXT
);

CREATE TABLE macro_indicators (
    macro_id                                   INTEGER PRIMARY KEY,
    country_code                               TEXT    NOT NULL,
    country_name                               TEXT,
    year                                       INTEGER NOT NULL UNIQUE,
    domestic_credit_to_private_sector_pct_of_gdp REAL,
    inflation_consumer_prices_annual_pct       REAL,
    lending_interest_rate_pct                  REAL,
    unemployment_total_pct_of_labour_force     REAL,
    avg_ocr_pct                                REAL,
    CHECK (year >= 1900)
);

CREATE TABLE borrowers (
    applicant_id          INTEGER PRIMARY KEY,
    person_age            REAL,
    person_income         REAL,
    person_home_ownership TEXT,
    person_emp_length     REAL,
    assessment_year       INTEGER,
    FOREIGN KEY (assessment_year) REFERENCES macro_indicators(year),
    CHECK (person_age IS NULL OR (person_age >= 18 AND person_age <= 100)),
    CHECK (person_income IS NULL OR person_income >= 0)
);

CREATE TABLE loans (
    loan_id              INTEGER PRIMARY KEY,
    applicant_id         INTEGER NOT NULL,
    source_id            INTEGER,
    loan_intent          TEXT,
    loan_grade           TEXT,
    loan_amnt            REAL,
    loan_int_rate        REAL,
    loan_percent_income  REAL,
    loan_to_income_check REAL,
    FOREIGN KEY (applicant_id) REFERENCES borrowers(applicant_id),
    FOREIGN KEY (source_id)    REFERENCES data_sources(source_id),
    CHECK (loan_amnt IS NULL OR loan_amnt >= 0),
    CHECK (loan_int_rate IS NULL OR loan_int_rate >= 0)
);

CREATE TABLE credit_history (
    history_id                  INTEGER PRIMARY KEY,
    applicant_id                INTEGER NOT NULL,
    cb_person_default_on_file   TEXT,
    cb_person_cred_hist_length  REAL,
    credit_score                REAL,
    FOREIGN KEY (applicant_id) REFERENCES borrowers(applicant_id)
);

CREATE TABLE risk_outcomes (
    outcome_id   INTEGER PRIMARY KEY,
    applicant_id INTEGER NOT NULL,
    default_flag INTEGER,
    risk_band    TEXT,
    FOREIGN KEY (applicant_id) REFERENCES borrowers(applicant_id),
    CHECK (default_flag IS NULL OR default_flag IN (0, 1))
);

CREATE INDEX idx_borrowers_assessment_year ON borrowers(assessment_year);
CREATE INDEX idx_loans_applicant_id        ON loans(applicant_id);
CREATE INDEX idx_loans_grade               ON loans(loan_grade);
CREATE INDEX idx_loans_source_id           ON loans(source_id);
CREATE INDEX idx_credit_history_applicant  ON credit_history(applicant_id);
CREATE INDEX idx_risk_outcomes_applicant   ON risk_outcomes(applicant_id);
CREATE INDEX idx_risk_band                 ON risk_outcomes(risk_band);

""")

conn.commit()
print("Database and schema created:", db_name)


## 6. Populate SQLite Tables

Each pandas DataFrame is inserted into its SQLite table using `to_sql` with `if_exists='append'`. The tables are inserted in dependency order: `data_sources` and `macro_indicators` first (no dependencies), then `borrowers`, then the tables that depend on `borrowers`.


In [ ]:
# Insert in dependency order
data_sources.to_sql("data_sources",    conn, if_exists="append", index=False)
macro_indicators.to_sql("macro_indicators", conn, if_exists="append", index=False)
borrowers.to_sql("borrowers",          conn, if_exists="append", index=False)
loans.to_sql("loans",                  conn, if_exists="append", index=False)
credit_history.to_sql("credit_history", conn, if_exists="append", index=False)
risk_outcomes.to_sql("risk_outcomes",  conn, if_exists="append", index=False)

conn.commit()
print("All tables populated.")

# Row count check
row_counts = pd.read_sql_query("""
SELECT 'borrowers'       AS table_name, COUNT(*) AS row_count FROM borrowers
UNION ALL
SELECT 'loans',                         COUNT(*) FROM loans
UNION ALL
SELECT 'credit_history',                COUNT(*) FROM credit_history
UNION ALL
SELECT 'risk_outcomes',                 COUNT(*) FROM risk_outcomes
UNION ALL
SELECT 'macro_indicators',              COUNT(*) FROM macro_indicators
UNION ALL
SELECT 'data_sources',                  COUNT(*) FROM data_sources;
""", conn)

print("\nRow counts by table:")
row_counts


## 7. SQL Analysis Queries

The following seven queries use multi-table JOINs to answer credit risk questions. Each query joins at least two tables and is explained before and after the output.


### Query 1 — Default Rate by Loan Grade

This query joins `loans` and `risk_outcomes` to calculate the average default rate, average loan amount and average interest rate for each loan grade. It helps answer Research Question 1 from the Assessment 4 notebook.


In [ ]:
q1 = pd.read_sql_query("""
SELECT
    l.loan_grade,
    COUNT(*)                             AS number_of_loans,
    ROUND(AVG(l.loan_amnt),    2)        AS avg_loan_amount,
    ROUND(AVG(l.loan_int_rate), 2)       AS avg_interest_rate,
    ROUND(AVG(r.default_flag) * 100, 2) AS default_rate_pct
FROM loans l
JOIN risk_outcomes r ON l.applicant_id = r.applicant_id
GROUP BY l.loan_grade
ORDER BY default_rate_pct DESC;
""", conn)
print("Query 1 — Default rate by loan grade:")
q1


**Interpretation:** Lower loan grades (E, F, G) are consistently associated with higher default rates and higher average interest rates. This confirms the credit risk gradient captured by the grading system.


### Query 2 — Default Rate by Home Ownership and Loan Purpose

This query joins three tables (`borrowers`, `loans`, `risk_outcomes`) to examine how home ownership combined with loan purpose relates to default risk. Only combinations with at least 10 borrowers are shown.


In [ ]:
q2 = pd.read_sql_query("""
SELECT
    b.person_home_ownership,
    l.loan_intent,
    COUNT(*)                             AS applicants,
    ROUND(AVG(b.person_income), 2)       AS avg_income,
    ROUND(AVG(l.loan_amnt),     2)       AS avg_loan_amount,
    ROUND(AVG(r.default_flag) * 100, 2) AS default_rate_pct
FROM borrowers b
JOIN loans l         ON b.applicant_id = l.applicant_id
JOIN risk_outcomes r ON b.applicant_id = r.applicant_id
GROUP BY b.person_home_ownership, l.loan_intent
HAVING COUNT(*) >= 10
ORDER BY default_rate_pct DESC, applicants DESC;
""", conn)
print("Query 2 — Default rate by home ownership and loan purpose (top 20):")
q2.head(20)


**Interpretation:** Renters taking loans for debt consolidation or personal purposes tend to show higher default rates. This segment may represent financially stretched borrowers who use credit to manage existing obligations.


### Query 3 — Risk Band Profile with Borrower Income and Interest Rate

This query joins `borrowers`, `loans` and `risk_outcomes` to build a financial profile for each risk band. It supports Research Question 3 on borrower segmentation.


In [ ]:
q3 = pd.read_sql_query("""
SELECT
    r.risk_band,
    COUNT(*)                                  AS applicants,
    ROUND(AVG(b.person_income),        2)     AS avg_income,
    ROUND(AVG(l.loan_amnt),            2)     AS avg_loan_amount,
    ROUND(AVG(l.loan_int_rate),        2)     AS avg_interest_rate,
    ROUND(AVG(l.loan_percent_income),  4)     AS avg_loan_pct_income
FROM borrowers b
JOIN loans         l ON b.applicant_id = l.applicant_id
JOIN risk_outcomes r ON b.applicant_id = r.applicant_id
GROUP BY r.risk_band
ORDER BY avg_interest_rate DESC;
""", conn)
print("Query 3 — Risk band profile:")
q3


**Interpretation:** High-risk borrowers pay higher interest rates and borrow a greater proportion of their income, which worsens their repayment capacity. Low-risk borrowers have higher incomes and lower loan-to-income ratios.


### Query 4 — Previous Default History and Current Default Rate

This query joins `credit_history`, `loans` and `risk_outcomes` to test whether having a previous default on file is associated with a higher current default rate.


In [ ]:
q4 = pd.read_sql_query("""
SELECT
    ch.cb_person_default_on_file,
    COUNT(*)                                    AS applicants,
    ROUND(AVG(ch.cb_person_cred_hist_length), 2) AS avg_credit_history_years,
    ROUND(AVG(l.loan_int_rate),               2) AS avg_interest_rate,
    ROUND(AVG(r.default_flag) * 100,          2) AS current_default_rate_pct
FROM credit_history ch
JOIN loans         l  ON ch.applicant_id = l.applicant_id
JOIN risk_outcomes r  ON ch.applicant_id = r.applicant_id
GROUP BY ch.cb_person_default_on_file
ORDER BY current_default_rate_pct DESC;
""", conn)
print("Query 4 — Default history vs current default rate:")
q4


**Interpretation:** Borrowers with a previous default on file ('Y') show a higher current default rate than those without ('N'). This confirms that credit history is a meaningful predictor of current repayment behaviour.


### Query 5 — Credit Risk by Assessment Year with Macroeconomic Indicators

This query joins all four borrower-related tables plus `macro_indicators` (five tables) to examine how credit risk compares across years alongside macroeconomic conditions.


In [ ]:
q5 = pd.read_sql_query("""
SELECT
    b.assessment_year,
    COUNT(*)                                               AS applicants,
    ROUND(AVG(r.default_flag) * 100,                    2) AS default_rate_pct,
    ROUND(AVG(l.loan_int_rate),                         2) AS avg_loan_interest_rate,
    ROUND(m.inflation_consumer_prices_annual_pct,       2) AS inflation_pct,
    ROUND(m.unemployment_total_pct_of_labour_force,     2) AS unemployment_pct,
    ROUND(m.lending_interest_rate_pct,                  2) AS nz_lending_rate_pct,
    ROUND(m.avg_ocr_pct,                                2) AS nz_ocr_pct
FROM borrowers b
JOIN loans           l  ON b.applicant_id = l.applicant_id
JOIN risk_outcomes   r  ON b.applicant_id = r.applicant_id
JOIN macro_indicators m ON b.assessment_year = m.year
GROUP BY b.assessment_year
ORDER BY b.assessment_year;
""", conn)
print("Query 5 — Credit risk by year with macroeconomic context:")
q5


**Interpretation:** The query shows how default rates compare across assessment years alongside New Zealand's macroeconomic conditions. Higher inflation and OCR years (2022–2023) represent a tighter credit environment. Note that the year assignment is illustrative (see Assessment 4 notebook).


### Query 6 — Highest Exposure Segments by Grade and Purpose

This query joins `loans` and `risk_outcomes` to identify which grade and purpose combinations represent the largest total loan exposure and highest default risk. This supports portfolio risk management decisions.


In [ ]:
q6 = pd.read_sql_query("""
SELECT
    l.loan_grade,
    l.loan_intent,
    r.risk_band,
    COUNT(*)                             AS number_of_loans,
    ROUND(SUM(l.loan_amnt),    2)        AS total_loan_exposure,
    ROUND(AVG(l.loan_amnt),    2)        AS avg_loan_amount,
    ROUND(AVG(r.default_flag) * 100, 2) AS default_rate_pct
FROM loans l
JOIN risk_outcomes r ON l.applicant_id = r.applicant_id
GROUP BY l.loan_grade, l.loan_intent, r.risk_band
HAVING COUNT(*) >= 5
ORDER BY total_loan_exposure DESC
LIMIT 20;
""", conn)
print("Query 6 — Top 20 highest exposure segments:")
q6


**Interpretation:** The segments with the largest total loan exposure are not always the highest default risk. A lender needs to consider both exposure size and default rate together. Debt consolidation loans in lower grades represent the most concerning combination of high exposure and high default risk.


### Query 7 — Highest Loan Burden Applicants with Data Source Reference

This query joins five tables including `data_sources` to list individual borrowers with the highest loan-to-income ratios, enriched with macroeconomic conditions and the source of the loan data.


In [ ]:
q7 = pd.read_sql_query("""
SELECT
    b.applicant_id,
    b.person_age,
    b.person_home_ownership,
    ROUND(b.person_income,          0) AS person_income,
    ROUND(l.loan_amnt,              0) AS loan_amnt,
    ROUND(l.loan_to_income_check,   3) AS loan_to_income_ratio,
    l.loan_grade,
    r.risk_band,
    b.assessment_year,
    ROUND(m.inflation_consumer_prices_annual_pct, 2) AS inflation_pct,
    ds.source_name
FROM borrowers b
JOIN loans            l  ON b.applicant_id = l.applicant_id
JOIN risk_outcomes    r  ON b.applicant_id = r.applicant_id
JOIN macro_indicators m  ON b.assessment_year = m.year
JOIN data_sources     ds ON l.source_id = ds.source_id
WHERE l.loan_to_income_check IS NOT NULL
ORDER BY l.loan_to_income_check DESC
LIMIT 15;
""", conn)
print("Query 7 — Top 15 borrowers by loan-to-income ratio (with data source):")
q7


**Interpretation:** Borrowers with the highest loan-to-income ratios are often in lower grades with higher risk bands. This confirms that over-leveraged borrowers represent concentrated credit risk. Including the data source name via the `data_sources` table demonstrates the FK relationship and traceability of the data.


## 8. Visualisations from SQL Results


In [ ]:
# Chart from Query 1 — default rate by loan grade
if len(q1) > 0:
    q1_plot = q1.dropna(subset=["loan_grade"]).sort_values("loan_grade")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(q1_plot["loan_grade"], q1_plot["default_rate_pct"], color="tomato", edgecolor="white")
    ax.set_title("Default Rate by Loan Grade (from SQLite Query)", fontsize=13)
    ax.set_xlabel("Loan Grade", fontsize=11)
    ax.set_ylabel("Default Rate (%)", fontsize=11)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()


In [ ]:
# Chart from Query 5 — default rate and macroeconomic indicators by year
if len(q5) > 0:
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.bar(q5["assessment_year"].astype(str), q5["default_rate_pct"],
            color="tomato", alpha=0.7, label="Default Rate (%)")
    ax1.set_xlabel("Assessment Year", fontsize=11)
    ax1.set_ylabel("Default Rate (%)", fontsize=11, color="tomato")
    ax2 = ax1.twinx()
    ax2.plot(q5["assessment_year"].astype(str), q5["inflation_pct"],
             marker="o", color="steelblue", label="Inflation (%)", linewidth=2)
    ax2.plot(q5["assessment_year"].astype(str), q5["unemployment_pct"],
             marker="s", color="seagreen", label="Unemployment (%)", linewidth=2, linestyle="--")
    ax2.set_ylabel("Macro Indicator (%)", fontsize=11)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    ax1.set_title("Default Rate vs Macroeconomic Indicators by Year", fontsize=13)
    plt.tight_layout()
    plt.show()


## 9. Create and Test Database Views

Two views are created as reusable virtual tables. Views save complex join logic so that users can query them like a normal table without repeating the SQL each time.

- **`vw_applicant_credit_profile`** — one complete joined profile per applicant combining borrower, loan, credit history and risk outcome data.
- **`vw_yearly_credit_macro_summary`** — yearly aggregated risk summary joined with macroeconomic indicators.


In [ ]:
cur.executescript("""
DROP VIEW IF EXISTS vw_applicant_credit_profile;
DROP VIEW IF EXISTS vw_yearly_credit_macro_summary;

CREATE VIEW vw_applicant_credit_profile AS
SELECT
    b.applicant_id,
    b.person_age,
    b.person_income,
    b.person_home_ownership,
    b.person_emp_length,
    b.assessment_year,
    l.loan_id,
    l.loan_intent,
    l.loan_grade,
    l.loan_amnt,
    l.loan_int_rate,
    l.loan_to_income_check,
    ch.cb_person_default_on_file,
    ch.cb_person_cred_hist_length,
    ch.credit_score,
    r.default_flag,
    r.risk_band
FROM borrowers b
JOIN loans          l  ON b.applicant_id = l.applicant_id
JOIN credit_history ch ON b.applicant_id = ch.applicant_id
JOIN risk_outcomes  r  ON b.applicant_id = r.applicant_id;

CREATE VIEW vw_yearly_credit_macro_summary AS
SELECT
    b.assessment_year,
    COUNT(*)                                               AS applicants,
    ROUND(AVG(l.loan_amnt),                             2) AS avg_loan_amount,
    ROUND(AVG(l.loan_int_rate),                         2) AS avg_loan_interest_rate,
    ROUND(AVG(r.default_flag) * 100,                    2) AS default_rate_pct,
    ROUND(m.inflation_consumer_prices_annual_pct,       2) AS inflation_pct,
    ROUND(m.unemployment_total_pct_of_labour_force,     2) AS unemployment_pct,
    ROUND(m.lending_interest_rate_pct,                  2) AS nz_lending_rate_pct,
    ROUND(m.avg_ocr_pct,                                2) AS nz_ocr_pct,
    ROUND(m.domestic_credit_to_private_sector_pct_of_gdp, 2) AS domestic_credit_pct_gdp
FROM borrowers b
JOIN loans            l  ON b.applicant_id = l.applicant_id
JOIN risk_outcomes    r  ON b.applicant_id = r.applicant_id
JOIN macro_indicators m  ON b.assessment_year = m.year
GROUP BY b.assessment_year;
""")

conn.commit()
print("Views created successfully.")


In [ ]:
# Test View 1
print("Testing vw_applicant_credit_profile (first 10 rows):")
pd.read_sql_query("""
SELECT *
FROM vw_applicant_credit_profile
LIMIT 10;
""", conn)


In [ ]:
# Test View 2
print("Testing vw_yearly_credit_macro_summary (all rows):")
pd.read_sql_query("""
SELECT *
FROM vw_yearly_credit_macro_summary
ORDER BY assessment_year;
""", conn)


## 10. Database Verification

Final checks to confirm all expected database objects (tables, views, indexes) were successfully created.


In [ ]:
# List all tables and views
print("Tables and views in database:")
pd.read_sql_query("""
SELECT name, type
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name;
""", conn)


In [ ]:
# List all indexes
print("Indexes in database:")
pd.read_sql_query("""
SELECT name, tbl_name
FROM sqlite_master
WHERE type = 'index'
ORDER BY tbl_name, name;
""", conn)


In [ ]:
# Summary count of database objects
print("Database object count by type:")
pd.read_sql_query("""
SELECT type, COUNT(*) AS number_created
FROM sqlite_master
WHERE type IN ('table', 'view', 'index')
GROUP BY type;
""", conn)


## 11. Interpretation of Findings

The SQL queries provide the following business-relevant observations:

- **Loan grade is the most useful risk grouping variable.** Queries 1 and 6 confirm that lower grades carry higher default rates and total exposure. A lender should apply more conservative lending criteria for grades E, F and G.
- **Home ownership and loan purpose create meaningful segments.** Query 2 shows that renters taking debt consolidation or personal loans default at higher rates. This segment deserves additional scrutiny during underwriting.
- **Previous default history is predictive of current default.** Query 4 confirms that borrowers with 'Y' on the credit bureau default flag have a higher current default rate. This is consistent with industry experience.
- **High loan-to-income borrowers are concentrated in lower grades.** Query 7 shows that the most over-leveraged borrowers tend to be in lower risk bands, creating compounding credit risk.
- **The database traceability via `data_sources` is useful.** Query 7 demonstrates that the data source for each loan can be traced via the FK relationship, which supports data governance.

**Limitation:** The `assessment_year` assignment in the borrower dataset is illustrative rather than based on real loan origination dates. In a production system, actual application dates and borrower geography would be needed before drawing causal conclusions from the macroeconomic join.


## 12. Close Database Connection


In [ ]:
conn.close()
print("SQLite connection closed.")


## Appendix — Files Produced by This Notebook

When run successfully from top to bottom, this notebook produces:

- `credit_risk_assignment3.sqlite` — SQLite database file with 6 tables, 7 indexes and 2 views.
- `credit_risk_schema.png` — schema diagram image embedded in the notebook.

Before submission, rerun the notebook from top to bottom and confirm all cell outputs are visible.
